In [5]:
import numpy as np
import tensorflow as tf
import joblib
import cv2
import mediapipe as mp
from ultralytics import YOLO
import math

# === Paths ===
VIDEO_PATH = "D:\\bha\\app\\mybacked\\neext\\input.mp4"
SCALER_PATH = "D:\\bha\\app\\mybacked\\neext\\scaler.save"
MODEL_PATH = "D:\\bha\\app\\mybacked\\neext\\final_model_attention.keras"

# === Load model and scaler ===
model = tf.keras.models.load_model(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

# === Load YOLO and MediaPipe ===
yolo = YOLO("yolov8n.pt")
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1, min_detection_confidence=0.5)

# === Utility: Calculate angle between 3 points ===
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    angle = np.arccos(np.clip(cosine, -1.0, 1.0))
    return np.degrees(angle)

# === Utility: Extract 6 joint angles ===
def extract_joint_angles(landmarks):
    joints = []
    # Right Elbow
    joints.append(calculate_angle(landmarks[12][:3], landmarks[14][:3], landmarks[16][:3]))
    # Left Elbow
    joints.append(calculate_angle(landmarks[11][:3], landmarks[13][:3], landmarks[15][:3]))
    # Right Shoulder
    joints.append(calculate_angle(landmarks[24][:3], landmarks[12][:3], landmarks[14][:3]))
    # Left Shoulder
    joints.append(calculate_angle(landmarks[23][:3], landmarks[11][:3], landmarks[13][:3]))
    # Right Knee
    joints.append(calculate_angle(landmarks[24][:3], landmarks[26][:3], landmarks[28][:3]))
    # Left Knee
    joints.append(calculate_angle(landmarks[23][:3], landmarks[25][:3], landmarks[27][:3]))
    return np.array(joints)

# === Step 1: Extract 10 annotated frames and features ===
def extract_features_from_video(video_path, max_frames=10):
    print("Extracting Features...")
    cap = cv2.VideoCapture(video_path)
    print("cv2 done")
    features = []
    frame_count = 0

    while cap.isOpened() and frame_count < max_frames:
        print(f"processing frame {frame_count+1} ...")
        ret, frame = cap.read()
        if not ret:
            break

        results = yolo(frame, verbose=False)
        if isinstance(results, list) and len(results) > 0:
            result = results[0]
            if hasattr(result, 'boxes') and result.boxes.xyxy.shape[0] > 0:
                x1, y1, x2, y2 = result.boxes.xyxy.cpu().numpy()[0].astype(int)
                cropped = frame[y1:y2, x1:x2]
                rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
                results_pose = pose.process(rgb)

                if results_pose.pose_landmarks:
                    landmarks = np.array(results_pose.pose_landmarks.landmark)
                    landmark_array = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in landmarks])
                    joint_angles = extract_joint_angles(landmark_array)  # shape (6,)
                    pose_feats = landmark_array.flatten()  # shape (132,)
                    full_feature = np.concatenate([pose_feats, joint_angles])  # 138
                    features.append(full_feature)
                    frame_count += 1

    cap.release()
    pose.close()

    while len(features) < 10:
        features.append(np.zeros(138))

    return np.array(features)[:10]

# === Step 2: Final tensor prep ===
def prepare_input_tensor(frames_10):
    if frames_10.shape[0] != 10:
        raise ValueError(f"Need 10 frames, got {frames_10.shape[0]}")

    velocity = np.gradient(frames_10[:, :99], axis=0)  # 99 = x,y,z
    final_features = np.concatenate([frames_10, velocity], axis=-1)  # (10, 237)
    final_scaled = scaler.transform(final_features)  # (10, 237)
    print("prepared input tensor")
    return np.expand_dims(final_scaled, axis=0)  # (1, 10, 237)

# === Step 3: Predict ===

def final_prediction(video_path):
    frames = extract_features_from_video(video_path)
    input_tensor = prepare_input_tensor(frames)
    print("Getting the prediction...")
    pred = model.predict(input_tensor)
    print("Prediction Done.")

    # === Display Result ===
    class_names = ['Good Technique', 'Low Arm', 'Poor Left Leg Block', 'Both Errors']
    predicted_class = np.argmax(pred)
    confidence = np.max(pred)

    print("\n🔍 Class-wise Probabilities:")
    for i, prob in enumerate(pred[0]):
        print(f"  {class_names[i]}: {prob:.3f}")

    print(f"\n✅ Predicted Class: {class_names[predicted_class]} | Confidence: {confidence:.4f}")

    result={
        "prediction": class_names[predicted_class],
        "confidence": confidence,
        "probabilities": {
                "Good Technique": pred[0][0],
                "Low Arm": pred[0][1],
                "Poor Left Leg Block": pred[0][2],
                "Both Errors": pred[0][3]
            }
    }

    return result

d:\bha\app\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
final_prediction(VIDEO_PATH)

Extracting Features...
cv2 done
processing frame 1 ...
processing frame 1 ...
processing frame 2 ...
processing frame 3 ...
processing frame 4 ...
processing frame 5 ...
processing frame 6 ...
processing frame 7 ...
processing frame 8 ...
processing frame 9 ...
processing frame 10 ...
prepared input tensor
Getting the prediction...
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Prediction Done.

🔍 Class-wise Probabilities:
  Good Technique: 0.029
  Low Arm: 0.029
  Poor Left Leg Block: 0.037
  Both Errors: 0.905

✅ Predicted Class: Both Errors | Confidence: 0.9050


{'prediction': 'Both Errors',
 'confidence': np.float32(0.9049924),
 'probabilities': {'Good Technique': np.float32(0.028888663),
  'Low Arm': np.float32(0.028794808),
  'Poor Left Leg Block': np.float32(0.037324224),
  'Both Errors': np.float32(0.9049924)}}

In [ ]:
pred

array([[   0.028889,    0.028795,    0.037324,     0.90499]], dtype=float32)

In [4]:
pred[0][1]

np.float32(0.028794808)